# 43 — Resource Allocation

**Engineering statement:** Operating regimes specify adaptive resource allocation.

Notebook 43 starts the second systems arc of the confidence-scheduled decoding repository. The first arc moved from mechanism to operating regime. This notebook moves from operating regime to controller policy.

The controller maps

\[
s_t \rightarrow \rho_t \rightarrow b_t \rightarrow a_t \rightarrow y_t
\]

where \(s_t\) is measured serving state, \(\rho_t\) is the active operating regime, \(b_t\) is the budget vector, \(a_t\) is the allocation action, and \(y_t\) is the observed serving outcome.


In [ ]:
# Setup

from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIG_DIR = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_resource_allocation"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 15,
    "legend.fontsize": 11,
})

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, width, height, text, fontsize=12, lw=1.7):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=lw,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=1.7, mutation_scale=14, connectionstyle="arc3"):
    patch = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=mutation_scale,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle
    )
    ax.add_patch(patch)
    return patch


## 1. Resource allocation flow

The notebook begins with the deployable chain:

```text
operating regime
→ adaptive controller
→ budget partition
→ verification scheduler
→ serving system
```

This is the minimal control path that Notebook 49 can turn into infrastructure.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Resource allocation flow for adaptive serving", pad=24)

labels = [
    "Operating\nregime",
    "Adaptive\ncontroller",
    "Budget\npartition",
    "Verification\nscheduler",
    "Serving\nsystem",
]
xs = np.linspace(0.08, 0.82, len(labels))
w, h, y = 0.14, 0.16, 0.55

for x, label in zip(xs, labels):
    rounded_box(ax, (x, y), w, h, label, fontsize=14)

for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1 + w, y + h/2), (x2, y + h/2))

rounded_box(
    ax, (0.28, 0.23), 0.44, 0.13,
    "The controller spends scarce resources according to the active regime.",
    fontsize=13
)

savefig("43_resource_flow.png")


## 2. Adaptive throughput gains

A fixed allocation rule is too blunt. Different regimes have different marginal returns to additional budget.


In [ ]:
budget = np.linspace(0.06, 1.0, 11)

series = {
    "balanced": 1.18 * (1 - np.exp(-4.2 * budget)),
    "compute-limited": 0.95 * (1 - np.exp(-2.2 * budget)),
    "memory-limited": 0.95 * (1 - np.exp(-3.0 * budget)),
    "latency-protected": 0.92 * (1 - np.exp(-5.0 * budget)) - 0.22 * budget**2 + 0.08,
}

fig, ax = plt.subplots(figsize=(12, 7))
for name, vals in series.items():
    ax.plot(budget, vals, marker="o", linewidth=2, label=name)

ax.set_title("Adaptive throughput gains depend on regime")
ax.set_xlabel("allocated resource budget")
ax.set_ylabel("throughput gain proxy")
ax.grid(True, alpha=0.35)
ax.legend(loc="upper left")
savefig("43_adaptive_gains.png")


## 3. Adaptive resource-allocation controller

This is the notebook centerpiece. The controller estimates state, classifies regime, selects policy, partitions resources, observes system state, and updates.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive resource-allocation controller", pad=20)

ax.text(
    0.5, 0.91,
    "system state → operating regime → controller policy → resource budget → serving outcome",
    ha="center", va="center", fontsize=14
)

# Main vertical path
main_nodes = [
    ("Incoming\nrequests", 0.78),
    ("State\nestimator", 0.66),
    ("Operating-regime\nclassifier", 0.54),
    ("Controller\npolicy", 0.42),
    ("Budget\nallocator", 0.30),
]
for label, y in main_nodes:
    rounded_box(ax, (0.38, y), 0.24, 0.08, label, fontsize=12)

for y1, y2 in [(0.78, 0.74), (0.66, 0.62), (0.54, 0.50), (0.42, 0.38)]:
    arrow(ax, (0.50, y1), (0.50, y2), lw=1.7)

# Budget channels
channels = [
    (0.10, "Compute"),
    (0.27, "Memory"),
    (0.44, "Verification"),
    (0.61, "Reserve"),
    (0.78, "Latency\nheadroom"),
]
for x, label in channels:
    rounded_box(ax, (x, 0.16), 0.15, 0.08, label, fontsize=11)
    arrow(ax, (0.50, 0.30), (x + 0.075, 0.24), lw=1.3)

# Serving and observation
rounded_box(ax, (0.38, 0.05), 0.24, 0.08, "Serving\nsystem", fontsize=12)
rounded_box(ax, (0.72, 0.05), 0.22, 0.08, "Observed\nsystem state", fontsize=11)
for x, _ in channels:
    arrow(ax, (x + 0.075, 0.16), (0.50, 0.13), lw=1.1)

arrow(ax, (0.62, 0.09), (0.72, 0.09), lw=1.5)
arrow(ax, (0.83, 0.13), (0.83, 0.70), lw=1.4)
arrow(ax, (0.83, 0.70), (0.62, 0.70), lw=1.4)

# State update side loop
rounded_box(ax, (0.08, 0.16), 0.18, 0.08, "State\nupdate", fontsize=11)
arrow(ax, (0.38, 0.34), (0.26, 0.20), lw=1.2)
arrow(ax, (0.17, 0.24), (0.38, 0.70), lw=1.2)

ax.text(
    0.5, 0.02,
    "Operating regimes specify controller policies; controller policies specify resource allocation.",
    ha="center", va="center", fontsize=13
)

savefig("43_controller_architecture.png")


## 4. Adaptive policy curves

Policy fractions vary continuously with available budget. Compute and verification fractions increase, while reserve decreases once the controller has enough headroom.


In [ ]:
b = np.linspace(0.06, 1.0, 11)

compute_frac = 0.14 + 0.58 * (1 - np.exp(-2.4 * b))
verify_frac  = 0.12 + 0.46 * (1 - np.exp(-3.2 * b))
memory_frac  = 0.27 + 0.13 * np.exp(-((b - 0.32) / 0.22) ** 2) - 0.05 * b
reserve_frac = np.maximum(0.08, 0.45 * np.exp(-4.2 * b))

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(b, compute_frac, marker="o", linewidth=2, label="compute fraction")
ax.plot(b, verify_frac, marker="o", linewidth=2, label="verification fraction")
ax.plot(b, memory_frac, marker="o", linewidth=2, label="memory fraction")
ax.plot(b, reserve_frac, marker="o", linewidth=2, label="reserve fraction")

ax.set_title("Adaptive policy curves allocate budget by regime")
ax.set_xlabel("available resource budget")
ax.set_ylabel("allocation fraction")
ax.set_ylim(0, 0.86)
ax.grid(True, alpha=0.35)
ax.legend(loc="upper right")
savefig("43_policy_curves.png")


## 5. Resource-allocation surface

The same compute or memory budget has different marginal value depending on the rest of the budget vector. A controller should operate on the surface, not assume uniform returns.


In [ ]:
x = np.linspace(0.1, 1.0, 120)
y = np.linspace(0.1, 1.0, 120)
X, Y = np.meshgrid(x, y)

Z = (1 - np.exp(-2.6 * X)) * (1 - np.exp(-2.2 * Y))
Z += 0.10 * np.exp(-((X - 0.86)**2 / 0.08 + (Y - 0.86)**2 / 0.08))
Z -= 0.06 * np.exp(-((X - 0.18)**2 / 0.02 + (Y - 0.18)**2 / 0.02))

points = {
    "compute-limited": (0.42, 0.70),
    "balanced": (0.62, 0.62),
    "latency-protected": (0.55, 0.50),
    "memory-limited": (0.72, 0.42),
}

fig, ax = plt.subplots(figsize=(11, 9))
cont = ax.contourf(X, Y, Z, levels=28)
for label, (px, py) in points.items():
    ax.scatter(px, py, s=120, label=label)
    ax.text(px + 0.02, py + 0.02, label, fontsize=12)

cbar = fig.colorbar(cont, ax=ax)
cbar.set_label("throughput proxy")
ax.set_title("Resource allocation surface for adaptive serving")
ax.set_xlabel("compute budget")
ax.set_ylabel("memory budget")
ax.set_xlim(0.1, 1.0)
ax.set_ylim(0.1, 1.0)
savefig("43_resource_surface.png")


## 6. Controller policy surface

The controller can map measured serving-state variables to a selected policy. Here the axes are GPU utilization and queue depth.


In [ ]:
gpu = np.linspace(0.1, 1.0, 150)
queue = np.linspace(0.1, 1.0, 150)
G, Q = np.meshgrid(gpu, queue)

# Policy IDs:
# 0 balanced, 1 compute-limited, 2 memory-limited, 3 latency-protected
policy = np.zeros_like(G, dtype=int)
policy[(G > 0.63) & (Q < 0.42)] = 2
policy[(G > 0.75) & (Q >= 0.20) & (Q < 0.68)] = 1
policy[(Q > 0.67) & (G < 0.72 + 0.20*(Q-0.67))] = 3
policy[(G < 0.58) & (Q < 0.67) & (((G-0.52)/0.30)**2 + ((Q-0.48)/0.23)**2 < 1.0)] = 0

cmap = ListedColormap(["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(
    policy, origin="lower", extent=[gpu.min(), gpu.max(), queue.min(), queue.max()],
    cmap=cmap, norm=norm, aspect="auto"
)

labels = ["balanced", "compute-\nlimited", "memory-\nlimited", "latency-\nprotected"]
coords = [(0.42, 0.35), (0.75, 0.30), (0.73, 0.72), (0.30, 0.84)]
for text, (tx, ty) in zip(labels, coords):
    ax.text(tx, ty, text, fontsize=13, ha="center", va="center")

cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["balanced", "compute-limited", "memory-limited", "latency-protected"])
cbar.set_label("selected policy")

ax.set_title("Controller policy surface from measurable serving state")
ax.set_xlabel("GPU utilization")
ax.set_ylabel("queue depth")
savefig("43_policy_surface.png")


## 7. Batch admission control

Admission control turns the current budget into a batch-level decision. The admitted batch should fit compute budget, memory budget, latency target, and reserve capacity.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Batch admission control under resource constraints", pad=24)

rounded_box(ax, (0.31, 0.77), 0.38, 0.10, "Admission depends on the current resource budget.", fontsize=14)

main = [
    ("Incoming\nrequests", 0.08, 0.54, 0.15, 0.12),
    ("Queue", 0.30, 0.54, 0.13, 0.12),
    ("Admission /\nallocation\ncontroller", 0.50, 0.52, 0.18, 0.16),
    ("Admitted\nbatch", 0.76, 0.54, 0.14, 0.12),
    ("Target\nverify", 0.93, 0.54, 0.10, 0.12),
]
for label, x0, y0, ww, hh in main:
    rounded_box(ax, (x0, y0), ww, hh, label, fontsize=12)

for (x1, y1, w1, h1), (x2, y2, w2, h2) in zip(
    [(m[1], m[2], m[3], m[4]) for m in main[:-1]],
    [(m[1], m[2], m[3], m[4]) for m in main[1:]]
):
    arrow(ax, (x1 + w1, y1 + h1/2), (x2, y2 + h2/2))

constraints = [
    ("compute\nbudget", 0.34),
    ("memory\nbudget", 0.47),
    ("latency\ntarget", 0.60),
    ("reserve\ncapacity", 0.73),
]
for label, cx in constraints:
    rounded_box(ax, (cx, 0.18), 0.11, 0.10, label, fontsize=10)
    arrow(ax, (cx + 0.055, 0.28), (0.59, 0.52), lw=1.3)

savefig("43_batch_admission_control.png")


## 8. Budget partitions

The controller changes the budget partition by regime. Balanced regimes spend broadly. Latency-protected regimes preserve reserve. Memory-limited regimes rebalance to control memory pressure.


In [ ]:
regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
components = ["compute", "memory", "verify", "reserve"]
data = np.array([
    [0.30, 0.25, 0.30, 0.15],
    [0.22, 0.30, 0.20, 0.28],
    [0.32, 0.18, 0.25, 0.25],
    [0.25, 0.22, 0.18, 0.35],
])

fig, ax = plt.subplots(figsize=(12, 7))
left = np.zeros(len(regimes))
for i, comp in enumerate(components):
    vals = data[:, i]
    bars = ax.barh(regimes, vals, left=left, label=comp)
    for j, bar in enumerate(bars):
        ax.text(left[j] + vals[j]/2, bar.get_y() + bar.get_height()/2,
                f"{comp}\n{vals[j]:.2f}", ha="center", va="center", fontsize=10)
    left += vals

ax.set_xlim(0, 1)
ax.set_title("Adaptive budget partitions by operating regime")
ax.set_xlabel("normalized resource budget")
ax.grid(True, axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=4)
savefig("43_budget_partition.png")


## 9. Allocation-policy phase diagram

The allocation policy changes across compute availability and memory availability. This controller map is intentionally coarse; it specifies scheduling behavior rather than a physical phase transition.


In [ ]:
x = np.linspace(0.1, 1.0, 140)
y = np.linspace(0.1, 1.0, 140)
X, Y = np.meshgrid(x, y)

# Policy IDs:
# 0 balanced, 1 compute-heavy, 2 memory-heavy, 3 verification-heavy, 4 latency-reserve
P = np.zeros_like(X, dtype=int)
P[(X < 0.45) & (Y > 0.42)] = 1
P[(Y < 0.42) & (X < 0.75)] = 4
P[(X > 0.72) & (Y < 0.45)] = 2
P[(Y > 0.73) & (X > 0.45)] = 3
balanced_island = (((X - 0.58) / 0.22) ** 2 + ((Y - 0.60) / 0.19) ** 2) < 1.0
P[balanced_island] = 0

cmap = ListedColormap(["#bcbd22", "#9467bd", "#1f77b4", "#17becf", "#2ca02c"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(P, origin="lower", extent=[0.1, 1.0, 0.1, 1.0], cmap=cmap, norm=norm, aspect="auto")

annotations = {
    "balanced": (0.58, 0.62),
    "compute-\nheavy": (0.27, 0.77),
    "memory-\nheavy": (0.82, 0.30),
    "verification-\nheavy": (0.58, 0.88),
    "latency-\nreserve": (0.30, 0.28),
}
for txt, (tx, ty) in annotations.items():
    ax.text(tx, ty, txt, ha="center", va="center", fontsize=13)

cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(["balanced", "compute-heavy", "memory-heavy", "verification-heavy", "latency-reserve"])
cbar.set_label("selected allocation policy")

ax.set_title("Allocation-policy phase diagram")
ax.set_xlabel("compute availability")
ax.set_ylabel("memory availability")
savefig("43_phase_diagram.png")


## 10. Repository roadmap

Notebook 43 starts the second systems arc. The first arc established the mechanism-to-regime story. The next arc moves from controllers toward adaptive infrastructure and distributed scheduling.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Resource allocation opens the second systems arc", pad=24)

# First arc
first = [
    ("00\nContext", 0.04),
    ("07\nVerification\nResource", 0.19),
    ("13\nConfidence\nScheduling", 0.34),
    ("17\nSemi-AR\nDrafting", 0.49),
    ("23\nThroughput\nObjective", 0.64),
    ("29\nHardware\nConstraints", 0.79),
    ("37\nOperating\nRegimes", 0.91),
]
w, h, y = 0.10, 0.12, 0.66
for label, x0 in first:
    rounded_box(ax, (x0, y), w, h, label, fontsize=10)
for (_, x1), (_, x2) in zip(first[:-1], first[1:]):
    arrow(ax, (x1 + w, y + h/2), (x2, y + h/2), lw=1.6)

ax.text(0.5, 0.56, "first arc: mechanism → operating regime", ha="center", va="center", fontsize=13)
ax.plot([0.06, 0.94], [0.51, 0.51], color="black", lw=1.4)

# Second arc
second = [
    ("43\nResource\nAllocation", 0.35),
    ("49\nAdaptive\nInfrastructure", 0.53),
    ("53\nDistributed\nScheduling", 0.71),
]
w2, h2, y2 = 0.13, 0.12, 0.28
for label, x0 in second:
    rounded_box(ax, (x0, y2), w2, h2, label, fontsize=10)
for (_, x1), (_, x2) in zip(second[:-1], second[1:]):
    arrow(ax, (x1 + w2, y2 + h2/2), (x2, y2 + h2/2), lw=1.6)

ax.text(
    0.5, 0.15,
    "second arc: operating regime → resource allocation → infrastructure",
    ha="center", va="center", fontsize=13
)

savefig("43_repository_arc.png")


## Key equations

Operating-regime classification:

\[
\rho_t=f(s_t)
\]

Regime-conditioned budget:

\[
b_t=\pi(\rho_t,s_t)
\]

Resource allocation action:

\[
a_t=A(b_t)
\]

Serving outcome:

\[
y_t=g(a_t)
\]

Closed-loop update:

\[
s_{t+1}=h(s_t,a_t,y_t).
\]


## Summary

Notebook 43 reframes adaptive serving as a resource-allocation controller.

The controller:
- reads the current operating regime,
- maps that regime to a budget policy,
- partitions compute, memory, verification, and reserve capacity,
- schedules verification under the current resource budget,
- observes the resulting serving state,
- updates the next allocation.

Notebook 49 can now move from adaptive resource allocation into adaptive infrastructure.


## Download notebook artifacts

Run the final cell to package Notebook 43 outputs for download.

The archive includes:

- this notebook,
- generated `43_*.png` figures,
- the manifest JSON.


In [ ]:
# Package Notebook 43 artifacts for download.

manifest = {
    "notebook": "43_resource_allocation_v5_release.ipynb",
    "title": "Resource Allocation",
    "engineering_statement": "Operating regimes specify adaptive resource allocation.",
    "figures": sorted([p.name for p in FIG_DIR.glob("43_*.png")]),
    "next_notebook": "49_adaptive_infrastructure.ipynb",
}

manifest_path = RESULTS / "43_resource_allocation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = RESULTS / "43_resource_allocation_v5_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Include figures.
    for p in sorted(FIG_DIR.glob("43_*.png")):
        zf.write(p, arcname=f"figures/{p.name}")

    # Include results.
    for p in sorted(RESULTS.glob("*")):
        if p.is_file() and p.resolve() != zip_path.resolve():
            zf.write(p, arcname=f"results/43_resource_allocation/{p.name}")

    # Include notebook if available in a typical location.
    candidates = [
        Path.cwd() / "43_resource_allocation_v5_release.ipynb",
        Path.cwd() / "43_resource_allocation.ipynb",
        REPO_ROOT / "notebooks" / "43_resource_allocation_v5_release.ipynb",
        REPO_ROOT / "notebooks" / "43_resource_allocation.ipynb",
    ]
    for nb_path in candidates:
        if nb_path.exists():
            zf.write(nb_path, arcname=f"notebooks/{nb_path.name}")
            break

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
